In [5]:
%load_ext autoreload
%autoreload 2

In [1]:
abc = [1, 2, 3, 4, 5, 6]

In [4]:
windows = 5

for i in range(len(abc) - windows + 1):
    print(abc[i:i+windows])

[1, 2, 3, 4, 5]
[2, 3, 4, 5, 6]


In [6]:
import torch
from PIL import Image
import numpy as np
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF

from transformers import ProcessorMixin
from transformers import MllamaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from transformers import TrainingArguments
from transformers import Trainer
from peft import LoraConfig, get_peft_model

from dall_e import map_pixels, unmap_pixels, load_model
from dall_e import Encoder, Decoder

from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation
from datasets import load_dataset, Dataset

from src.agents.tools.transmission import EncodingTool, DecodingTool
from src.agents.tools.segmentation import SemanticSegmentationTool

from torchmetrics import JaccardIndex

from src.kitti_tracking import KittiDataset
from src.kitti_tracking_hf import KittiHFIterableDataset
from arc_trainer import ArcTrainer
from arc_utils import ArcProcessor, ExtendedLMHead, ExtendEmbedding

c:\Users\ngoak\anaconda3\envs\agentic\lib\site-packages\torchreid\reid\metrics\rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(


In [7]:
processor = ArcProcessor.from_llama_vision(
    ckpt_path="./checkpoints",
    model_id="meta-llama/Llama-3.2-11B-Vision"
)

In [13]:
n_steps, n_pred_steps = 8, 0
dataset_builder = KittiHFIterableDataset(
	root_dir="C:/Users/ngoak/data/kitti_tracking",
	split="training",
	n_steps=n_steps,
	n_pred_steps=n_pred_steps,
	transform=T.Compose([
	    T.Resize((240, 640)),
	])
)

hf_dataset = dataset_builder.to_hf_dataset()

for i, sample in enumerate(hf_dataset):
    if len(sample['rgb']) < 8:
        print(i, len(sample['rgb']), len(sample['depth']))
    if i > 600: break
    # break

583 7 7
584 6 6
585 5 5
586 4 4


In [10]:
i

7860

In [ ]:
# root_dir = "C:/Users/ngoak/data/kitti_tracking"
# n_steps, n_pred_steps = 8, 1

# dataset_builder = KittiHFIterableDataset(
# 	root_dir="C:/Users/ngoak/data/kitti_tracking",
# 	split="training",
# 	n_steps=n_steps,
# 	n_pred_steps=n_pred_steps,
# 	transform=T.Compose([
# 	    T.Resize((240, 640)),
# 	])
# )

# hf_dataset = dataset_builder.to_hf_dataset()

# sample = next(iter(hf_dataset))
# print(sample.keys())

# inputs = processor(
#     prompts=["person"],
#     images=sample["rgb"],
#     text="",
#     # mask=
# )

In [4]:
# add more tokens to the vocabulary
# load tokenizers
model_id = "meta-llama/Llama-3.2-11B-Vision"

model = MllamaForConditionalGeneration.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
)
model.lm_head = ExtendedLMHead.from_llama("./checkpoints", model)
model.language_model.embed_tokens = ExtendEmbedding.from_llama("./checkpoints", model)
# model.resize_token_embeddings(len(processor.tokenizer))
model.language_model.embed_tokens.base_embedding.requires_grad = False
model.language_model.embed_tokens.extra_embedding.requires_grad = False


model.lm_head.base_head.requires_grad = False
model.lm_head.extra_head.requires_grad = True

lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    lora_dropout=0.1,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'down_proj', 'gate_proj', 'up_proj', 
        # "embed_tokens", "lm_head",
    ],
    use_dora=True, # optional DoRA 
    init_lora_weights="gaussian"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# llm_processor = AutoProcessor.from_pretrained(model_id)

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

trainable params: 31,416,320 || all params: 10,741,483,043 || trainable%: 0.2925


In [ ]:
def preprocess_fn(sample):
	images = sample["rgb"]   # list of PIL or arrays
	prompts = ["object"]     # or your actual prompts

	processed = processor(
		prompts=prompts,
		images=images,
		return_tensors="pt",
		padding=True,
		truncation=True
	)
	_pixel_values = processed.get("pixel_values", None)
	
	return {
		"input_ids": processed["input_ids"][0],
		"attention_mask": processed["attention_mask"][0],
		"pixel_values": _pixel_values[0] if _pixel_values is not None else None,
		"labels": processed["input_ids"][0],  # causal LM
		"aspect_ratio_ids": processed["aspect_ratio_ids"][0],  # causal LM
		"aspect_ratio_mask": processed["aspect_ratio_mask"][0],  # causal LM
	}

root_dir = "C:/Users/ngoak/data/kitti_tracking"
n_steps, n_pred_steps = 8, 1

dataset_builder = KittiHFIterableDataset(
	root_dir="C:/Users/ngoak/data/kitti_tracking",
	split="training",
	n_steps=n_steps,
	n_pred_steps=n_pred_steps,
	transform=T.Compose([
		T.Resize((120, 320)),
	])
)

hf_dataset = dataset_builder.to_hf_dataset()
hf_dataset = hf_dataset.map(preprocess_fn, batched=False)   # important since you loop per sample)
training_args = TrainingArguments(
	max_steps=1000,
	output_dir='./results',
	logging_dir='./logs',
	gradient_checkpointing=True,
	per_device_train_batch_size=1,
	per_device_eval_batch_size=1,
	# num_train_epochs=1,
	# logging_steps=10,
	# save_total_limit=2,
	# max_steps=1,
	# disable_tqdm=False,       # enable progress bar
    logging_strategy="steps",
    logging_steps=10,          # show every step
    logging_first_step=True,  # show step 0/1
)
arc_trainer = Trainer(
	# ckpt_path="",
	model=model,
	args=training_args,
	train_dataset=hf_dataset,
)
arc_trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.


8
8


KeyboardInterrupt: 

In [ ]:

args = TrainingArguments(
    num_train_epochs=2,
    remove_unused_columns=False,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_steps=2,
    learning_rate=2e-5,
    weight_decay=1e-6,
    adam_beta2=0.999,
    logging_steps=250,
    save_strategy="no",
    optim="adamw_hf",
    push_to_hub=False,
    save_total_limit=1,
    bf16=True,
    output_dir="./lora",
    dataloader_pin_memory=False,
)
trainer = Trainer(
    model=model,
    train_dataset=ds,
    data_collator=process,
    args=args
)

In [ ]:
# VQ-VAE
def preprocess(img: Image.Image) -> torch.Tensor:
	img = torch.unsqueeze(T.ToTensor()(img), 0)
	return map_pixels(img)  # (1 - 2 * 0.1) * x + 0.1

def vq_encode(image, model, dev):
	x = preprocess(image).to(dev)
	z_logits = model(x.to(dev))
	z = torch.argmax(z_logits, axis=1)
	
	return z

def vq_decode(codes, model):
	z = F.one_hot(codes, num_classes=enc.vocab_size).permute(0, 3, 1, 2).float()

	x_stats = model(z).float()
	x_rec = unmap_pixels(torch.sigmoid(x_stats[:, :3]))
	x_rec = T.ToPILImage(mode='RGB')(x_rec[0])

	return x_rec

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
enc: Encoder = load_model("./checkpoints/encoder.pkl", device)
dec: Decoder = load_model("./checkpoints/decoder.pkl", device)

In [ ]:
sem_segm_tool = SemanticSegmentationTool(model_name="CIDAS/clipseg-rd64-refined")

In [ ]:
# load "codewords" from VAE
new_tokens = [
    f"<vq_{i}>" for i in range(enc.blocks[-1].conv.w.shape[0])
] + ["<begin_mask>", "<end_mask>"]

processor.tokenizer.add_tokens(new_tokens)
model.resize_token_embeddings(len(processor.tokenizer))

In [ ]:
# load dataset for fine-tuning
root_dir = "C:/Users/ngoak/data/kitti_tracking"
n_steps, n_pred_steps = 16, 3


train_ds = KittiDataset(
	root_dir, "training",
	n_steps, n_pred_steps,
	inp_transforms=T.Compose([
		T.Resize((240, 640)),
	])
)
for sample_dict in train_ds:
	rgb_images, depth_images = sample_dict['rgb'], sample_dict['depth']
	
	# rgb_code =
	# rgb_code_str =
	break

In [ ]:
iou_metric = JaccardIndex(task="binary")
tgt_class = ["person"]
prompt = "Goal"
for image in rgb_images:
	_prompt = "<|image|>"

	sem_mask = sem_segm_tool(image, tgt_class)
	z = vq_encode(sem_mask.convert("RGB"), enc, device)
	_z = z[0].cpu().numpy()
	mask_s = "<begin_mask>" + "".join([
		f"<vq_{_z[i, j]}>"
		for i in range(_z.shape[0])
		for j in range(_z.shape[1])
	]) + "<end_mask>"

	_sem_mask = torch.round(torch.tensor(np.array(sem_mask)) / 255)
	patch = Image.new("RGB", (50, 50), (255, 255, 255))
	sem_mask.paste(patch, (100, 100))
	_sem_mask2 = torch.round(torch.tensor(np.array(sem_mask)) / 255)
	iou = 1 - iou_metric(_sem_mask2, _sem_mask)
	
	_prompt = "<|image|>" +\
	f"Thought: To ensure novel information is included in the transmission, I generate mask that reduces curiosity.\n" +\
	f"Action: {mask_s}\n" +\
	f"Observation: {iou:6f}\n" 

	print(iou)

	break

In [ ]:
gt = "<|image1|>" +\
    f"<|begin_of_text|> The curious mask that marks regions that might posibly contain {','.join(tgt_class)}"# +\
    # f"{mask_s}\n"

gt

In [ ]:
_prompt

In [ ]:
sem_mask

In [ ]:
sem_mask

In [ ]:
x = preprocess(rgb_images[0]).to(model.device)
z_logits = enc(x.to(device))
z = torch.argmax(z_logits, axis=1)
z.shape
display(T.ToPILImage(mode='RGB')(x[0]))

In [ ]:
_z = z[0].cpu().numpy()
mask_s = "<begin_mask>" + "".join([
    f"<vq_{_z[i, j]}>"
    for i in range(_z.shape[0])
    for j in range(_z.shape[1])
]) + "<end_mask>"


In [ ]:
processor.tokenizer(mask_s, return_tensors="pt", ).input_ids

In [ ]:
z = vq_encode(rgb_images[0], enc, device)

x_rec = vq_decode(z, dec)

display(x_rec)

In [ ]:
import requests


url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/0052a70beed5bf71b92610a43a52df6d286cd5f3/diffusers/rabbit.jpg"
image = Image.open(requests.get(url, stream=True).raw)

# prompt = "<|image|><|begin_of_text|>If I had to write a haiku for this one"
# inputs = processor(image, prompt, return_tensors="pt").to(model.device)

# output = model.generate(**inputs, max_new_tokens=30)